# Producción Agricola (Preparación)

En este analisis de datos usaremos como base la Producción Agricola Municipal a partir de datos del DGSIAP. Para este proyecto se crearon clases dedicadas a la gestión de archivos. En esta libreta veremos su uso en un analisis de datos pequeño.

Para la extracción de los datos contruimos un diccionario maestro que almacena toda la información relevante.
* Diccionario de Datos
* Metadatos
* Los propios datos.

Se carga una vez y lo tenemos disponible para su uso en cualquier analisis

In [1]:
from scripts.config import source_links
from scripts.dgsiap import dict_master
from scripts.Files import WebFile
import yarl
dgsiap = dict_master()


## Descarga

Como estos archivos serán descargados de internet serán trabajados mediante la clase WebFile

In [2]:
def parse_link(url, base):
    y_url= yarl.URL(url)
    link = base.with_query(y_url.query)
    return link

dgsiap_api = source_links['DGSIAP']['download']

data = dgsiap['Municipal']['Datasets'][0]

filelink = parse_link(data, dgsiap_api)

print(filelink)
dataset2025 = WebFile(filelink, "DGSIAP")
dataset2025.set_filename()
dataset2025.download()

https://nube.agricultura.gob.mx/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2025
El nombre del archivo encontrado es Cierre_agricola_mun_2025.csv
El archivo ya fue descargado. Se encuentra en /data/pure/DGSIAP


Como podemos ver. La clase admite agregar elementos como la fuente y el enlace de descarga. Así gestionamos de forma más comoda la descarga de los mismos. 

Luego, notemos que podemos definir un documento principal, el diccionario de datos, y archivos relacionados, los datos asociados. Se almacenaran como un WebTree. Como estamos trabajando con datos municipales crearemos un WebTree que almacene los datos municipales extraidos del DGSIAP

In [3]:
from scripts.Files import INEGITree

dg_trees = {v:dgsiap[v] for v in dgsiap.keys()}
mun_tree = dg_trees['Municipal']

mun_metadata = mun_tree['Metadata']['info']['download']
mun_datasets = mun_tree['Datasets']


mun = INEGITree("DGSIAP", parse_link(mun_metadata, dgsiap_api), [
    parse_link(data, dgsiap_api) for data in mun_datasets])
mun.metadata(mun_tree['Metadata'])
mun.download()

El nombre del archivo encontrado es Diccionario_agricola_2003_a_2025.xlsx
El archivo ya fue descargado. Se encuentra en /data/pure/DGSIAP
Archivo Descargado 1/24
El nombre del archivo encontrado es Cierre_agricola_mun_2025.csv
El archivo ya fue descargado. Se encuentra en /data/pure/DGSIAP
Archivo Descargado 2/24
El nombre del archivo encontrado es Cierre_agricola_mun_2024.csv
El archivo ya fue descargado. Se encuentra en /data/pure/DGSIAP
Archivo Descargado 3/24
El nombre del archivo encontrado es Cierre_agricola_mun_2023.csv
El archivo ya fue descargado. Se encuentra en /data/pure/DGSIAP
Archivo Descargado 4/24
El nombre del archivo encontrado es Cierre_agricola_mun_2022.csv
El archivo ya fue descargado. Se encuentra en /data/pure/DGSIAP
Archivo Descargado 5/24
El nombre del archivo encontrado es Cierre_agricola_mun_2021.csv
El archivo ya fue descargado. Se encuentra en /data/pure/DGSIAP
Archivo Descargado 6/24
El nombre del archivo encontrado es Cierre_agricola_mun_2020.csv
El archi

Despues de descargar todos los archivos del libro, esto porque la clase es un conjunto de archivos WebFile que admite la descarga. Debemos comenzar con el procesamiento.

### Carga

Ahora sigue el proceso de poder acceder a ellos para su uso.

In [4]:
# Podemos acceder a la información de los archivos desde el WebTree.
print(mun.rootfile.filename)

for file in mun.branchfiles:
    print(file.filename)

Diccionario_agricola_2003_a_2025.xlsx
Cierre_agricola_mun_2025.csv
Cierre_agricola_mun_2024.csv
Cierre_agricola_mun_2023.csv
Cierre_agricola_mun_2022.csv
Cierre_agricola_mun_2021.csv
Cierre_agricola_mun_2020.csv
Cierre_agricola_mun_2019.csv
Cierre_agricola_mun_2018.csv
Cierre_agricola_mun_2017.csv
Cierre_agricola_mun_2016.csv
Cierre_agricola_mun_2015.csv
Cierre_agricola_mun_2014.csv
Cierre_agricola_mun_2013.csv
Cierre_agricola_mun_2012.csv
Cierre_agricola_mun_2011.csv
Cierre_agricola_mun_2010.csv
Cierre_agricola_mun_2009.csv
Cierre_agricola_mun_2008.csv
Cierre_agricola_mun_2007.csv
Cierre_agricola_mun_2006.csv
Cierre_agricola_mun_2005.csv
Cierre_agricola_mun_2004.csv
Cierre_agricola_mun_2003.csv


### Encoding
Como podemos ver. Tenemos archivos .csv y .xlsx. Lo ideal fuera que el INEGITree (Un webtree adaptado a las bases de datos del INEGI) pudiera preparar los archivos para su futuro procesamiento. Para ello WebFile arregla un problema común de los .csv. La codificación usada. 

La función load se encarga de dos funciones.
* Detectar el encoding más probable.
* Preparar el .csv con el nuevo encoding para poder leerlo con pandas.

In [5]:
mun.load()

Los archivos estan cargados


Una vez cargados podemos llamarlos mediante la función open del propio WebFile

In [6]:
datadic = mun.rootfile.open()
datadic

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Conformación y descripción de la tabla de info...,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Consecutivo\nen la base,Nombre\nde campo,Tipo,Long.,Rangos\nválidos,Descripción,Observaciones
9,1,Anio,N,4,{CCCC},Año de la información,Corresponde a la referencia temporal de los da...


## Cambio de Etapa

Todos los archivos se encuentran en /data/pure/DGSIAP. Esto porque 
* Son los datos al ser descargados. (/data/pure)
* Los datos provienen del DGSIAP. (/DGSIAP)

Sin embargo no queremos estar procesando los archivos en cada llamada. Entonces usaremos la carpeta /raw para guardar la información con los minimos cambios posibles. Es decir, que puedan abrirse y darles un nombre más comodo para el uso.

Durante la carga arreglamos el problema de encoding en los .csv. Actuamente todos los archivos se pueden abrir. Entonces nos queda preparar el saldo de /pure a /raw para comenzar con el procesamiento de los datos, esto es la preparación de la etapa de la ingesta de datos.

### Diccionario de Datos .xlsx

El trabajo es de traslado y cambio de nombre. Sencillo pero debemos mantener la coherencia. Para ello usaremos el diccionario maestro. Revisando el inicio, 

```{python}
dg_trees = {v:dgsiap[v] for v in dgsiap.keys()}
mun_tree = dg_trees['Municipal']

mun_metadata = mun_tree['Metadata']['info']['download']
mun_datasets = mun_tree['Datasets']


mun = INEGITree("DGSIAP", parse_link(mun_metadata, dgsiap_api), [
    parse_link(data, dgsiap_api) for data in mun_datasets])
mun.metadata(mun_tree['Metadata'])
mun.download()
```

Al construir el arbol indicamos el tipo de los datos. Entonces usaremos eso para darles nombre. 


In [7]:
print("Nombre del archivo:",mun.rootfile.filename)
print("Ubicación en la carpeta pure:\n", mun.rootfile.ubication)
metadata = mun.metadata  # Podemos dar el nombre nosotros. Pero lo ideal es que lo proporcione la propia información extraida.
print(mun.metadata)

Nombre del archivo: Diccionario_agricola_2003_a_2025.xlsx
Ubicación en la carpeta pure:
 /home/dusty/Documents/Repositories/proyecto_agronomia/data/pure/DGSIAP/Diccionario_agricola_2003_a_2025.xlsx
{'metadata': {'title': 'Metadatos Principales', 'state': 'Actualizado', 'lastdate': '12 agosto 2026', 'range': 'Cobertura 2003 - 2025', 'size': 'Nivel Municipal'}, 'info': {'title': 'Diccionario de datos', 'description': 'Consulte cada nombre de campo, rangos válidos y la descripción de los campos que se utilizan en nuestras bases de datos.', 'download': '/index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D7'}}


Ahora. Vamos a guardar el archivo que construimos. Usando la información de la pagina vamos a llamar a los diccionarios mediante usando la categoria y el tamaño de los datos

In [8]:
mun.rootfile.prepare(name="Diccionario_ProdAgricola_mun")
for file in mun.branchfiles:
    file.prepare()

# Clase File

Todo esto podriamos hacerlo mediante la clase WebFile. Pero los archivos de la clase raw no fueron descargados. Entonces usaremos la clase File para su gestión. Entonces usaremos la función WebFile.prepare para crear un archivo de la clase TreeFile para su uso en la siguiente etapa.

In [9]:
from scripts.Files import FileTree

root_file = mun.rootfile.prepare(name="Diccionario_ProdAgricola_mun")
branch_fs = [file.prepare() for file in mun.branchfiles]

TO_PROCESS = FileTree(root = root_file, branch=branch_fs, source=mun.source)


Finalmente lo guardaremos como preload para no tener que hacer todo el proceso de nuevo. 
**IMPORTANTE** Se debe seguir todo el proceso de la libreta para que funiona de forma adecuada. 
Igualmente se añadirá un script que haga todo esto en automatico.

In [10]:
from scripts.config import ROOT
import pickle

# Se guarda aqui para no tener que procesar todo de nuevo.
with open(ROOT / "preload" / "DGSIAP_PREP.pkl", "wb") as pretoraw:
    pickle.dump(TO_PROCESS, pretoraw)
